## Import libraries

In [6]:
import pandas as pd
import numpy as np

# Show all columns
pd.set_option('display.max_columns', None)

## Load the dataset

In [7]:
merged_df = pd.read_csv(r'./data/merged.csv', low_memory=False)

Let's take a quick look at the dataset:

In [4]:
merged_df

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Day,CompetitionOpen
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,0,2015,7,31,82.0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",2015,7,31,92.0
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",2015,7,31,103.0
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,0.0,0.0,0,2015,7,31,70.0
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,0.0,0.0,0,2015,7,31,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
844333,682,2,2013-01-01,3375,566,1,0,a,1,b,a,150.0,9.0,2006.0,0,0.0,0.0,0,2013,1,1,76.0
844334,733,2,2013-01-01,10765,2377,1,0,a,1,b,b,860.0,10.0,1999.0,0,0.0,0.0,0,2013,1,1,159.0
844335,769,2,2013-01-01,5035,1248,1,0,a,1,b,b,840.0,NaN,NaN,1,48.0,2012.0,"Jan,Apr,Jul,Oct",2013,1,1,0.0
844336,948,2,2013-01-01,4491,1039,1,0,a,1,b,b,1430.0,NaN,NaN,0,0.0,0.0,0,2013,1,1,0.0


## Train-Test split

*Note: This is a time-series data so we can't do normal train-test split.*

In [8]:
# change the Dtype of Date column
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

In [9]:
# set the cutoff
cutoff = merged_df['Date'].max() - pd.Timedelta(weeks=6)

In [10]:
train = merged_df[merged_df['Date'] <= cutoff]
test = merged_df[merged_df['Date'] > cutoff]

In [11]:
X_train = train.drop('Sales', axis=1)
X_test = test.drop('Sales', axis=1)
y_train = train['Sales']
y_test = test['Sales']

Let's decide Input and Traget columns:

In [12]:
input_cols = ['DayOfWeek', 'Open',
              'Promo', 'StateHoliday', 'SchoolHoliday',
              'StoreType', 'Assortment', 'CompetitionDistance',
              'CompetitionOpen','Promo2', 'Promo2SinceWeek',
              'Promo2SinceYear', 'Year', 'Month', 'Day']

target_cols = 'Sales'

Let's separate `categorical` and `numerical` columns from input columns for `Encoding` and `Scaling` respectively,
note that we can treat binary categorical columns(0/1) as numeric columns *(Because with only two values, the “distance between values” problem disappears)*:

In [13]:
cat_cols = ['DayOfWeek', 'StoreType', 'Assortment', 'StateHoliday']

num_cols = ['Open', 'Promo', 'SchoolHoliday',
            'CompetitionDistance', 'CompetitionOpen', 'Promo2', 'Promo2SinceWeek',
            'Promo2SinceYear', 'Year', 'Month', 'Day']

## Preprocessing

#### Encoding categorical columns

In [14]:
from sklearn.preprocessing import OneHotEncoder

In [15]:
encoder = OneHotEncoder(sparse_output=False)

In [16]:
X_train_encoded = encoder.fit_transform(X_train[cat_cols])
X_test_encoded = encoder.transform(X_test[cat_cols])

#### Scaling numerical columns

In [17]:
from sklearn.preprocessing import StandardScaler

In [18]:
scaler = StandardScaler()

In [19]:
X_train_scaled = scaler.fit_transform(X_train[num_cols])
X_test_scaled = scaler.transform(X_test[num_cols])

#### Combine Encoded and Scaled columns

In [20]:
X_train_final = np.hstack([X_train_encoded, X_train_scaled])
X_test_final = np.hstack([X_test_encoded, X_test_scaled])

Let's check their shape before we go further:

In [18]:
X_train_final.shape, X_test_final.shape

((804056, 29), (40282, 29))

## Untuned Random Forest

In [21]:
from sklearn.ensemble import RandomForestRegressor

In [28]:
untuned_rf = RandomForestRegressor(random_state=42, n_jobs=-1)

In [29]:
# training
untuned_rf.fit(X_train_final, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [30]:
# predicting
untuned_rf_preds = untuned_rf.predict(X_test_final)

Let's evaluate our untuned model using `MAE`, and `R-Squared` from `sklearn`:

In [31]:
from sklearn.metrics import mean_absolute_error, r2_score

In [32]:
untuned_rf_mae = mean_absolute_error(y_test, untuned_rf_preds)
print('Untuned Random Forest MAE:', untuned_rf_mae)

Untuned Random Forest MAE: 789.4327721821082


Our untuned **Random forest** model is off by about **€789.43** (in either direction).

In [25]:
y_test_mean = y_test.mean()
print('y_test_mean:', y_test_mean)

y_test_mean: 6978.638051735266


Our untuned **Random Forest**'s MAE is about `11%` of the **average daily sales**.\
To put it simply: if average daily sales were 100, the model would typically predict somewhere around 89 to 111.

In [ ]:
untuned_rf_r2 = r2_score(y_test, untuned_rf_preds)
print('Untuned Random Forest R-Sqauared:', untuned_rf_r2)

Untuned Random Forest R-Sqauared: 0.8367099411855363


Our untuned model is able to explain **83.67%** of the variance in daily sales.

**Next:** Let's check feature importnace our untuned **Random Forest** but first let's get all the features:

In [27]:
# Features from encoding
encoded_features = encoder.get_feature_names_out()

In [28]:
# Features from scaling
scaled_features = scaler.get_feature_names_out()

In [29]:
# concatenate features from encoding and scaling
all_features = np.concatenate([encoded_features, scaled_features])

In [ ]:
pd.DataFrame({
    'Feature': all_features,
    'Importance': untuned_rf.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

,Feature,Importance
21,CompetitionDistance,0.366099
19,Promo,0.134081
22,CompetitionOpen,0.110770
28,Day,0.054087
25,Promo2SinceYear,0.047693
27,Month,0.047648
24,Promo2SinceWeek,0.040232
0,DayOfWeek_1,0.033727
8,StoreType_b,0.026326
7,StoreType_a,0.019237


So according to our untuned random forest model, the most important feature is **`CompetitionDistance`**.

## Untuned XGBoost

In [34]:
from xgboost import XGBRegressor

In [35]:
untuned_xgb = XGBRegressor(random_state=42, n_jobs=-1)

In [36]:
# training
untuned_xgb.fit(X_train_final, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [38]:
# prediction
untuned_xgb_preds = untuned_xgb.predict(X_test_final)

Now, Let's evaluate our untuned model using the same matrics as before:

In [39]:
untuned_xgb_mae = mean_absolute_error(y_test, untuned_xgb_preds)
print("Untuned XGBoost MAE:", untuned_xgb_mae)

Untuned XGBoost MAE: 1104.4725341796875


Our untuned **XGBoost** model is off by about **€1104.47**, which is even worse than our untuned **Random forest** model (€789.43).

In [36]:
y_test_mean = y_test.mean()
print('y_test_mean:', y_test_mean)

y_test_mean: 6978.638051735266


Our untuned **XGBoost**'s MAE is about `15.83%` of the **average daily sales**.

In [41]:
untuned_xgb_r2 = r2_score(y_test, untuned_xgb_preds)
print("Untuned XGBoost R-Squared:", untuned_xgb_r2)

Untuned XGBoost R-Squared: 0.7293480038642883


Untuned **XGBoost** is able to explain `73%` of vriance in daily sales, though untuned **Random Forest** was `83.67%`.

**Next:** Let's check feature importnace our untuned **XGBoost**:

In [ ]:
pd.DataFrame({
    'Feature': all_features,
    'Importance': untuned_xgb.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

,Feature,Importance
8,StoreType_b,0.169333
19,Promo,0.119308
0,DayOfWeek_1,0.084342
23,Promo2,0.064947
25,Promo2SinceYear,0.061946
11,Assortment_a,0.060220
21,CompetitionDistance,0.051665
24,Promo2SinceWeek,0.040255
9,StoreType_c,0.038929
10,StoreType_d,0.037100


Interesting, our untuned **XGBoost** is prioritizing `Assortment_b` and `Promo`as the 2nd most importnat feature.

---

## Random Forest Hyperparameter Tuning

From `sklearn` we'll use `RandomizedSearchCV` (since the dataset is large – *8,04,056 rows x 29 columns*) to find the best parameters for\
**Random Forest** and since we're dealing with time-series data we'll use `TimeSeriesSplit` for splits:

In [39]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

Let's choose hyperparameters and its possible value that could improve our baseline model:

In [40]:
param_grid = {
    'n_estimators': [100, 120, 200],
    'max_depth': [30, 40, 50],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': [0.7, 0.8, 0.9, 1.0],
    'max_samples': [ 0.5, 0.6, 0.7]
}

Let's decide number of splits for `RandomizedSearchCV`:

In [41]:
tscv = TimeSeriesSplit(n_splits = 5)

Let's configure `RandomizedSearchCV`:

In [42]:
rf_random_search = RandomizedSearchCV(
    estimator = RandomForestRegressor(random_state=42, n_jobs=-1), 
    param_distributions = param_grid,
    n_iter = 10,
    scoring = 'neg_mean_absolute_error',
    cv=tscv,
    n_jobs= 1,
    verbose=2
)

AS stated before, our dataset is large so it would take forever to fit `RandomizedSearchCV`. To tackle this issue\
let's use only 50% of the entire training data:

In [43]:
# time-series aware sampling, not random
X_train_sample = X_train_final[:400000]
y_train_sample = y_train.iloc[:400000]

Let's fit `RandomizedSearchCV` on sampled data:

In [44]:
rf_random_search.fit(X_train_sample, y_train_sample)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END max_depth=30, max_features=1.0, max_samples=0.5, min_samples_leaf=1, n_estimators=120; total time=   5.3s
[CV] END max_depth=30, max_features=1.0, max_samples=0.5, min_samples_leaf=1, n_estimators=120; total time=  11.1s
[CV] END max_depth=30, max_features=1.0, max_samples=0.5, min_samples_leaf=1, n_estimators=120; total time=  16.8s
[CV] END max_depth=30, max_features=1.0, max_samples=0.5, min_samples_leaf=1, n_estimators=120; total time=  25.1s
[CV] END max_depth=30, max_features=1.0, max_samples=0.5, min_samples_leaf=1, n_estimators=120; total time=  34.5s
[CV] END max_depth=50, max_features=1.0, max_samples=0.5, min_samples_leaf=5, n_estimators=200; total time=   5.9s
[CV] END max_depth=50, max_features=1.0, max_samples=0.5, min_samples_leaf=5, n_estimators=200; total time=  14.9s
[CV] END max_depth=50, max_features=1.0, max_samples=0.5, min_samples_leaf=5, n_estimators=200; total time=  22.6s
[CV] END max_depth=

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [30, 40, ...], 'max_features': [0.7, 0.8, ...], 'max_samples': [0.5, 0.6, ...], 'min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that ca

Let's extract best hyperParameters of `Random Forest` and its best value for each:

In [45]:
print("RandomSearch best Parameters:", rf_random_search.best_params_)
print("RandomSearch MAE:", -rf_random_search.best_score_)

RandomSearch best Parameters: {'n_estimators': 100, 'min_samples_leaf': 1, 'max_samples': 0.7, 'max_features': 0.9, 'max_depth': 40}
RandomSearch MAE: 985.8615544032485


Let's train our model on best parametrs we got from `RandomSearch`:

In [49]:
tuned_rf = RandomForestRegressor(random_state=42,
                                 n_jobs=-1,
                                 n_estimators= 100,
                                 min_samples_leaf= 2,
                                 max_samples = 0.7,
                                 max_features = 1.0,
                                 max_depth = 40
                                 )

In [50]:
tuned_rf.fit(X_train_final, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",40
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [51]:
tuned_rf_preds = tuned_rf.predict(X_test_final)

In [52]:
tuned_rf_mae = mean_absolute_error(y_test, tuned_rf_preds)
print("Tuned Random Forest MAE:", tuned_rf_mae)

Tuned Random Forest MAE: 776.1719389107618


In [53]:
tuned_rf_r2 = r2_score(y_test, tuned_rf_preds)
print("Tuned Random Forest R-Squared:", tuned_rf_r2)

Tuned Random Forest R-Squared: 0.8397803763910818


**Conclusion**

*NOTE: The randomized searches ran on a 400,000-row subsample of the training set because of memory limits.*

| Model | MAE | R² |
|---|---|---|
| Untuned (default hyperparameters) | €789 | 0.837 |
| Tuned (9 combinations, 5-fold TimeSeriesSplit) | €776 | 0.839 |

**Result:** tuning gave a small improvement: MAE fell by about €13 (roughly 1.6%) and R² rose by 0.002.


**What happened along the way:**\
My first search space contained only constrained forests (`max_depth` 10 to 25, `min_samples_leaf` 5 to 20, `max_features` 0.3 to 0.5,
`max_samples` 0.3 to 0.5). Every model in it was worse than the untuned (best MAE about €1,141), because it excluded the defaults:
unlimited depth, `min_samples_leaf=1`, all features per split. Once I searched near the default values, the tuned model matched and
slightly beat the untuned. So for this data, a Random Forest is best left close to its defaults, and restricting it costs accuracy.

---

## XGBoost Hyperparameter Tuning

Let's find the best values for the XGBoost hyperparameters that could improve the performance:

In [ ]:
param_grid = {
    'n_estimators': [600, 800, 1000, 1200],
    'learning_rate': [0.2, 0.3, 0.4],
    'max_depth': [8, 10, 12, 15],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5]
}

In [ ]:
xgb_random_search = RandomizedSearchCV(
    estimator = XGBRegressor(random_state=42, n_jobs=-1), 
    param_distributions = param_grid,
    n_iter = 10,
    scoring = 'neg_mean_absolute_error',
    cv=tscv,
    n_jobs= 1,
    verbose=2
)

In [ ]:
xgb_random_search.fit(X_train_sample, y_train_sample)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END colsample_bytree=0.9, learning_rate=0.2, max_depth=15, min_child_weight=1, n_estimators=800, subsample=0.8; total time=  24.0s
[CV] END colsample_bytree=0.9, learning_rate=0.2, max_depth=15, min_child_weight=1, n_estimators=800, subsample=0.8; total time=  32.9s
[CV] END colsample_bytree=0.9, learning_rate=0.2, max_depth=15, min_child_weight=1, n_estimators=800, subsample=0.8; total time=  40.1s
[CV] END colsample_bytree=0.9, learning_rate=0.2, max_depth=15, min_child_weight=1, n_estimators=800, subsample=0.8; total time=  50.1s
[CV] END colsample_bytree=0.9, learning_rate=0.2, max_depth=15, min_child_weight=1, n_estimators=800, subsample=0.8; total time=  56.2s
[CV] END colsample_bytree=0.9, learning_rate=0.2, max_depth=12, min_child_weight=1, n_estimators=1200, subsample=0.9; total time=  13.9s
[CV] END colsample_bytree=0.9, learning_rate=0.2, max_depth=12, min_child_weight=1, n_estimators=1200, subsample=0.9; tota

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.7, 0.8, ...], 'learning_rate': [0.2, 0.3, ...], 'max_depth': [8, 10, ...], 'min_child_weight': [1, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies 

In [ ]:
print("RandomSearch best Parameters:", xgb_random_search.best_params_)
print("RandomSearch MAE:", -xgb_random_search.best_score_)

RandomSearch best Parameters: {'subsample': 0.9, 'n_estimators': 1200, 'min_child_weight': 1, 'max_depth': 12, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
RandomSearch MAE: 1042.3086669921875


Let's train the model on RandomSearch best Parameters:

In [59]:
tuned_xgb = XGBRegressor(random_state=42,
                         n_jobs=-1,
                         subsample=0.9,
                         n_estimators=800,
                         min_child_weight=1,
                         max_depth=10,
                         learning_rate=0.2,
                         colsample_bytree=0.8)

In [60]:
tuned_xgb.fit(X_train_final, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [61]:
tuned_xgb_preds = tuned_xgb.predict(X_test_final)

In [62]:
tuned_xgb_mae = mean_absolute_error(y_test, tuned_xgb_preds)
print("Tuned XGBoost MAE:", tuned_xgb_mae)

Tuned XGBoost MAE: 778.6041870117188


In [63]:
tuned_xgb_r2 = r2_score(y_test, tuned_xgb_preds)
print("Tuned XGBoost R-Squared:", tuned_xgb_r2)

Tuned XGBoost R-Squared: 0.835636556148529


**Conclusion**

*NOTE: The randomized searches ran on a 400,000-row subsample of the training set because of memory limits.*

Tuned XGBoost over 8 rounds of randomized search (TimeSeriesSplit cross-validation) and compared it with the untuned XGBoost and with the Random Forest from earlier. All models use the same features, training data and test set.

| Model | Test MAE | Test R² |
|---|---|---|
| XGBoost untuned (defaults) | €1,104 | 0.73 |
| XGBoost, round 3 (worst) | €1,726 | 0.40 |
| XGBoost, round 5 | €959 | 0.784 |
| XGBoost, round 7 | €806 | 0.831 |
| **XGBoost, round 8 (best)** | **€779** | **0.836** |
| Random Forest (tuned) | €776 | 0.839 |

Best parameters (round 8):\
`n_estimators=800`, `max_depth=10`, `learning_rate=0.2`, `subsample=0.9`, `colsample_bytree=0.8`, `min_child_weight=1`.

**Result:** tuning cut XGBoost's test MAE from €1,104 to €779, about 30%. R² rose from 0.73 to 0.836. Unlike the Random Forest, where tuning gave almost nothing, XGBoost's defaults were clearly far from the best settings for this data.

**What drove the improvement:**
- **Rounds 1 to 5** used shallow trees (depth 5 to 7) and only 100 to 200 trees. Raising the learning rate helped each time, and round 3 (learning rate 0.02 with 150 trees) was the worst result. I read this as the model having too few trees to learn at a low rate, but I did not compare training error with test error, so this is an interpretation, not a measured fact.
- **Rounds 6 to 8** used deeper trees (depth 10 to 12) and more trees (300 to 800), and MAE fell from about €960 to €779. Several settings changed between rounds, so I cannot say which one contributed most.

**XGBoost versus Random Forest:** the two end up essentially tied (MAE: €779 vs €776, R²: 0.836 vs 0.839). The €3 gap is far smaller than the uncertainty in these estimates, so I do not claim either model is better.

**How much to trust these numbers:**
- I ran 8 rounds and adjusted the search after seeing test results, so the final test MAE is somewhat optimistic. A cleaner process would choose settings by CV score only and touch the test set once.
- CV MAE (about €1,039) is well above test MAE (€779) in every round. I attribute this to Subsampling (used only 400k training data, not the entire training data) and `TimeSeriesSplit` on it.

---

## Feature importance on tuned models

In [ ]:
# Tuned Random Forest
pd.DataFrame({
    'Feature': all_features,
    'Importance': tuned_rf.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

,Feature,Importance
21,CompetitionDistance,0.370099
19,Promo,0.137798
22,CompetitionOpen,0.108824
28,Day,0.049229
25,Promo2SinceYear,0.048803
27,Month,0.045157
24,Promo2SinceWeek,0.040730
0,DayOfWeek_1,0.034417
8,StoreType_b,0.026966
7,StoreType_a,0.019352


Just like untuned Random Forest, Tuned Random Forest as well picked `CompetitionDistance` as the most important and `Promo`\
as the 2nd most important feature.

In [ ]:
# Tuned XGBoost
pd.DataFrame({
    'Feature': all_features,
    'Importance': tuned_xgb.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

,Feature,Importance
8,StoreType_b,0.220994
19,Promo,0.117644
21,CompetitionDistance,0.084549
6,DayOfWeek_7,0.063823
25,Promo2SinceYear,0.048504
23,Promo2,0.043657
9,StoreType_c,0.042284
13,Assortment_c,0.039034
11,Assortment_a,0.035813
7,StoreType_a,0.033388


In case of XGBoost, just like the untuned XGBoost, Tuned XGBoost as well picked `StoreType_b` as the first most important and `Promo`\
as the 2nd most important feature.

Our both model in both of the cases (Untuned and Tuned) agreed on one point that our `Promo` increases our sales.

---
---